# Issue #6: Segment all evaluation subsets with YAP (v3)

Run all cells top to bottom. Everything is idempotent: the Drive cache is
scrubbed of poisoned entries (not deleted), so previously completed real
segmentations are reused and only the missing work is redone.

In [ ]:
from getpass import getpass
token = getpass('GitHub PAT: ')
%cd /content
!rm -rf /content/NLP-Final-
!git clone https://{token}@github.com/AdonZahavi/NLP-Final-.git /content/NLP-Final-
%cd /content/NLP-Final-
!git checkout issue-6-segment-subsets
!git pull
# sanity: the crash-safe script MUST be present
!grep -c 'YapServerDown' segmentation/segment_subsets.py

In [ ]:
# Mount Drive cache
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nlp_final_cache
!rm -rf /content/NLP-Final-/cache
!ln -s /content/drive/MyDrive/nlp_final_cache /content/NLP-Final-/cache
!ls -la /content/NLP-Final-/cache/

In [ ]:
# SCRUB the cache: drop entries identical to raw tokenization (poisoned
# fallbacks from the buggy runs), keep genuine segmentations.
import json, re, hashlib, os
TOK = re.compile(r"[א-ת0-9a-zA-Z\"'׳״]+|[^\s]")
def h(t): return hashlib.sha256(t.encode('utf-8')).hexdigest()

raw_forms = {}
for f, fields in [('sentiment_500', ['text']), ('nli_884', ['premise', 'hypothesis']),
                  ('qa_500', ['context', 'question'])]:
    for r in map(json.loads, open(f'/content/NLP-Final-/data/subsets/{f}.jsonl', encoding='utf-8')):
        for fld in fields:
            t = ' '.join(str(r[fld]).split())
            raw_forms[h(t)] = ' '.join(TOK.findall(t))

cache_path = '/content/NLP-Final-/cache/segment_cache.jsonl'
if os.path.exists(cache_path):
    kept, dropped = [], 0
    for line in open(cache_path, encoding='utf-8'):
        try:
            e = json.loads(line)
        except json.JSONDecodeError:
            continue
        if raw_forms.get(e['h']) == e['seg']:
            dropped += 1
        else:
            kept.append(line)
    open(cache_path, 'w', encoding='utf-8').writelines(kept)
    print(f'kept {len(kept)} real segmentations, dropped {dropped} poisoned entries')
else:
    print('no cache yet — starting fresh')

In [ ]:
# Raw sentiment TSVs (needed by gold_check.py; data/raw is gitignored)
%cd /content/NLP-Final-
!mkdir -p data/raw
!wget -q -O data/raw/token_test.tsv https://github.com/omilab/Neural-Sentiment-Analyzer-for-Modern-Hebrew/raw/master/data/token_test.tsv
!wget -q -O data/raw/morph_test.tsv https://github.com/omilab/Neural-Sentiment-Analyzer-for-Modern-Hebrew/raw/master/data/morph_test.tsv
!wc -l data/raw/*.tsv

In [ ]:
# Build YAP (proven recipe)
!apt-get update -qq && apt-get install -y -qq golang-go bzip2 > /dev/null 2>&1
import os
os.environ['GOPATH'] = '/content/gopath'
os.environ['GO111MODULE'] = 'off'

%cd /content
!rm -rf /content/gopath
!mkdir -p /content/gopath/src
!git clone -q https://github.com/OnlpLab/yap.git /content/gopath/src/yap
%cd /content/gopath/src/yap
!bunzip2 -k data/*.bz2
!git clone -q --depth 1 https://github.com/gorilla/mux.git vendor/github.com/gorilla/mux
!rm -rf vendor/github.com/gorilla/mux/.git
!mkdir -p vendor/gopkg.in
!git clone -q --depth 1 --branch v2 https://github.com/go-yaml/yaml.git vendor/gopkg.in/yaml.v2
!rm -rf vendor/gopkg.in/yaml.v2/.git
!ln -sf data/bgulex/bgupreflex_withdef.utf8.hr .
!ln -sf data/bgulex/bgulex.utf8.hr .
!go build -o /content/gopath/src/yap/yap_bin .
!ls -la /content/gopath/src/yap/yap_bin

In [ ]:
# YAP server manager: start (or restart) and wait until it answers
import subprocess, time, urllib.request, json

yap_proc = None

def start_yap(max_wait_rounds=60):
    global yap_proc
    if yap_proc is not None:
        try:
            yap_proc.kill(); yap_proc.wait()
        except Exception:
            pass
    logf = open('/content/yap.log', 'a')
    yap_proc = subprocess.Popen(
        ['./yap_bin', 'api'], cwd='/content/gopath/src/yap',
        stdout=logf, stderr=subprocess.STDOUT,
    )
    req = urllib.request.Request(
        'http://localhost:8000/yap/heb/joint',
        data=json.dumps({'text': 'שלום  '}).encode(),
        headers={'Content-Type': 'application/json'},
    )
    for attempt in range(max_wait_rounds):
        if yap_proc.poll() is not None:
            print(f'YAP EXITED (code {yap_proc.returncode}). Log tail:')
            print(open('/content/yap.log').read()[-2000:])
            return False
        try:
            with urllib.request.urlopen(req, timeout=120):
                print(f'YAP ready (attempt {attempt+1})')
                return True
        except Exception:
            time.sleep(10)
    print('YAP never became ready')
    return False

start_yap()

In [ ]:
# Smoke test — v2 script output MUST show '[OK]' and 'missing fields'
%cd /content/NLP-Final-
!python segmentation/segment_subsets.py --limit 3 --task sentiment

In [ ]:
# FULL RUN with auto-recovery: on abort, restart YAP and resume from cache
import subprocess as sp

for round_no in range(1, 16):
    print(f'===== segmentation round {round_no} =====')
    ret = sp.call(['python', 'segmentation/segment_subsets.py'], cwd='/content/NLP-Final-')
    if ret == 0:
        print('SEGMENTATION COMPLETE ✔')
        break
    print(f'run exited with code {ret} — restarting YAP and resuming...')
    if not start_yap():
        print('could not restart YAP; check the log above')
        break
else:
    print('gave up after 15 rounds — inspect /content/yap.log')

In [ ]:
# HARD validation: token-expansion ratio (real segmentation ≈ 1.2–1.4, raw = 1.0)
import json
for f in ['sentiment_500', 'nli_884', 'qa_500']:
    recs = [json.loads(l) for l in open(f'data/subsets/segmented/{f}.jsonl', encoding='utf-8')]
    raw_toks = seg_toks = 0
    for r in recs:
        for k, v in list(r.items()):
            if k.endswith('_seg'):
                raw_toks += len(str(r[k[:-4]]).split())
                seg_toks += len(v.split())
    ratio = seg_toks / raw_toks
    print(f, f'expansion ratio: {ratio:.2f}', '✔' if ratio > 1.15 else '✘ CONTAINS RAW DATA — re-run the loop cell')

In [ ]:
!pip install -q pandas
%cd /content/NLP-Final-
!python segmentation/gold_check.py

In [ ]:
!head -60 data/subsets/segmented/qc_sample.md

In [ ]:
# Commit ONLY after the expansion-ratio cell shows three ✔
%cd /content/NLP-Final-
!git config user.email "orna.zahavi1@gmail.com" && git config user.name "Or Zahavi"
!git add data/subsets/segmented/
!git commit -m "Issue #6 v3: fully segmented subsets (validated expansion ratios)"
!git push origin issue-6-segment-subsets